# GEHCAI-MAPSMR

This notebook serves as a guided tour of the [GEHCAI-MAPSMR](https://registry.opendata.aws/gehcai-mapsmr) dataset. The dataset contains a small collection of de-identified MRI volumes in NRRD format along with [3D Slicer](https://www.slicer.org/) markup annotations (ROI bounding boxes) in JSON format, contributed by GE HealthCare. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

MAPSMR (Multi‑Anatomy Post‑Surgical Magnetic Resonance Dataset) is the first multi‑organ, post‑surgical MRI benchmark dataset focused on organ absence and altered anatomy after common abdominal and pelvic surgeries. The dataset includes cases such as cholecystectomy, prostatectomy, nephrectomy, colectomy, hepatectomy, and related procedures, with annotations identifying surgically absent organs and post‑treatment anatomical changes. MAPS‑MR fills a critical gap in medical imaging AI by enabling evaluation of models that must perform reliably on patients with non‑standard, surgically modified anatomy.

### Dataset organization

At the top level of the `gehcai-mapsmr` S3 bucket, there are two key prefixes:

1. **`nrrd/`** — Contains 38 de-identified 3D abdominal MRI volumes in [NRRD](http://teem.sourceforge.net/nrrd/format.html) (Nearly Raw Raster Data) format. Each file is named with a unique anonymized hex identifier (e.g., `01D049541FB4.nrrd`).
2. **`markup/`** — Contains 38 corresponding annotation files in [3D Slicer Markup JSON](https://slicer.readthedocs.io/en/latest/developer_guide/modules/markups.html) format. Each annotation file defines a 3D ROI (Region of Interest) bounding box around the anatomical region of interest, and shares the same identifier as its paired NRRD image (e.g., `01D049541FB4.mrk.json`).

There is a 1:1 correspondence between image files and annotation files — every NRRD volume has exactly one matching markup JSON file. A markup file might have more than 1 bounding box annotation (e.g. left and right kidney). 

First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# pynrrd >= 1.1.0
# numpy >= 1.26.0
# matplotlib >= 3.10.3
# polars >= 1.30.0

# Import the libraries required for this notebook
# Built-ins
import io
import json
from pprint import pprint

# Installed libraries
import boto3
import nrrd
import numpy as np
import polars
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

Next, we will define the location of our dataset, create our boto3 S3 client, and list the top-level prefixes in our S3 bucket.

In [ ]:
# Location of the S3 bucket for this dataset
bucket = "gehcai-mapsmr"
prefix = "" # The dataset is organized with all files at the top level of the bucket, so no prefix is needed. If there were a prefix, it would be specified here (e.g. "data/").

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests.
# Here we set the signature version to unsigned, which is required for public buckets.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Print the items in the top-level prefixes
for item in s3.list_objects_v2(Bucket=bucket, Delimiter='/')['CommonPrefixes']:
    print(item['Prefix'])

We see there are two top-level prefixes: `markup/` for annotation files and `nrrd/` for MRI volumes. Let's list the contents of each.

In [ ]:
# List the NRRD image files
nrrd_objects = s3.list_objects_v2(Bucket=bucket, Prefix=prefix + 'nrrd/')['Contents']
nrrd_files = [obj for obj in nrrd_objects if obj['Key'].endswith('.nrrd')]
print(f"Number of NRRD image files: {len(nrrd_files)}")

# List the markup annotation files
markup_objects = s3.list_objects_v2(Bucket=bucket, Prefix=prefix + 'markup/')['Contents']
markup_files = [obj for obj in markup_objects if obj['Key'].endswith('.mrk.json')]
print(f"Number of markup annotation files: {len(markup_files)}")


### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

This dataset contains two data formats:

#### 1. NRRD (Nearly Raw Raster Data) — MRI Volumes

[NRRD](http://teem.sourceforge.net/nrrd/format.html) is a file format designed for scientific visualization and medical image computing. Each `.nrrd` file in this dataset is a 3D abdominal MRI volume with:

- **3 dimensions** (width × height × slices) with varying resolutions across scans
- **LPS (Left-Posterior-Superior) coordinate system**, standard in medical imaging
- **gzip compression** for efficient storage
- **Spatial metadata** including voxel spacing (space directions) and volume origin

NRRD was chosen because it:
- Is a standard in the medical imaging community, particularly for use with [3D Slicer](https://www.slicer.org/)
- Stores image data with full spatial metadata in a single file
- Supports efficient compression for large volumetric data
- Is well supported by Python via the [`pynrrd`](https://github.com/mhe/pynrrd) library, and by tools like [3D Slicer](https://www.slicer.org/), [ITK](https://itk.org/), and [SimpleITK](https://simpleitk.org/)

#### 2. 3D Slicer Markup JSON — Annotations

Each `.mrk.json` file follows the [3D Slicer Markups JSON schema](https://raw.githubusercontent.com/slicer/slicer/master/Modules/Loadable/Markups/Resources/Schema/markups-schema-v1.0.3.json#) and defines:

- A 3D **ROI (Region of Interest)** bounding box of type "Box"
- **Center coordinates** (in mm) and **box dimensions** (in mm) for the region of interest
- An **LPS coordinate system** consistent with the NRRD images
- Control point positions for visualization

These annotations were created using [3D Slicer](https://www.slicer.org/), a free, open-source software platform for medical image visualization and analysis. The markup files can be loaded directly in 3D Slicer for interactive viewing, or parsed as standard JSON in Python.

<!-- #### AWS Services

Services such as [Amazon SageMaker](https://aws.amazon.com/sagemaker/) can be used to train machine learning models (e.g., object detection or segmentation models) on this dataset. [Amazon Athena](https://aws.amazon.com/athena/) can be used to query the annotation metadata at scale. -->

### Example data download and usage

Let us download and inspect one NRRD volume and its corresponding markup annotation file.

In [ ]:
# Pick an example sample to examine. 
sample_id = nrrd_files[10]['Key'].split('/')[-1].replace('.nrrd', '')
print(f"Sample ID: {sample_id}") 
# This patient (ID#: 61E2ACB60713) has gone through "cholecystectomy" surgery, which is the removal of the gallbladder. 
# The gallbladder is visible in the scan as a dark sac-like structure in the upper right abdomen, but it has been surgically removed in this patient, so it appears as an empty space.

# Download and load the NRRD volume
nrrd_key = f"{prefix}nrrd/{sample_id}.nrrd"
nrrd_body = s3.get_object(Bucket=bucket, Key=nrrd_key)['Body'].read()
fh = io.BytesIO(nrrd_body)
header = nrrd.read_header(fh)
data = nrrd.read_data(header, fh, io.BytesIO(nrrd_body))

print(f"\n--- NRRD Header ---")
for key, value in header.items():
    print(f"  {key}: {value}")

print(f"\n--- Volume Info ---")
print(f"  Shape: {data.shape}")
print(f"  Data type: {data.dtype}")
print(f"  Min value: {data.min()}")
print(f"  Max value: {data.max()}")

Now let's load the corresponding markup annotation file to see its ROI bounding box definition.

In [ ]:
# Download and load the corresponding annotation
markup_key = f"{prefix}markup/{sample_id}.mrk.json"
markup_response = s3.get_object(Bucket=bucket, Key=markup_key)
markup_data = json.loads(markup_response['Body'].read())

# Display the annotation structure
pprint(markup_data)

Let's extract the key fields from the annotation: the ROI center, size, and coordinate system.

In [ ]:
# Extract key ROI properties from the first markup
roi = markup_data['markups'][0]
print(f"ROI Type:           {roi['roiType']}")
print(f"Coordinate System:  {roi['coordinateSystem']}")
print(f"Coordinate Units:   {roi['coordinateUnits']}")
print(f"Center (LPS, mm):   {roi['center']}")
print(f"Size (mm):          {roi['size']}")
print(f"Orientation:        {roi['orientation']}")

### Overlay annotation on the MR image. 
Simplest way would be do drag/drop matching nrrd/mrk.json files onto Slicer3D software. Here we are using matplotlib and numpy. 

First, let's visualize a central axial slice from our sample MRI volume with the ROI bounding box overlaid.

In [ ]:
# Display the central axial slice with ROI bounding box overlay
mid_slice = data.shape[2] // 2
slice_img = data[:, ::-1, mid_slice].T  
# Transpose for correct orientation to match the typical radiological view and visualization with matplotlib. 
# Here we are using LPS convention and flipping the y-axis to match the typical radiological view (origin at bottom left, x=right-left, y=posterior-anterior). 
# if you are using other visualization libraries or conventions, you may need to adjust the orientation and flipping accordingly.

# Extract spatial info from NRRD header
space_directions = np.array(header['space directions'])
origin = np.array(header['space origin'])
voxel_spacing = np.abs(np.diag(space_directions))  # Approximate for axis-aligned volumes

# Convert ROI center and size from physical (mm) to voxel coordinates
roi_center = np.array(roi['center'])
roi_size = np.array(roi['size'])

# Convert physical coordinates to voxel indices
# Note: LPS coordinate system; NRRD space directions define the mapping
roi_min_phys = roi_center - roi_size / 2
roi_max_phys = roi_center + roi_size / 2

roi_min_vox = (roi_min_phys - origin) / voxel_spacing
roi_max_vox = (roi_max_phys - origin) / voxel_spacing
roi_min_vox[1] = data.shape[1] - roi_min_vox[1]  # Flip y-axis for correct orientation
roi_max_vox[1] = data.shape[1] - roi_max_vox[1]  # Flip y-axis for correct orientation

# Ensure correct ordering (min < max)
roi_min_vox, roi_max_vox = np.minimum(roi_min_vox, roi_max_vox), np.maximum(roi_min_vox, roi_max_vox)

# Plot the axial slice
fig, ax = plt.subplots(1, 1, figsize=(8, 8), dpi=100, facecolor='white')
ax.imshow(slice_img, cmap='gray', origin='lower')

# Draw ROI rectangle on the axial slice (x = col, y = row)
from matplotlib.patches import Rectangle
rect = Rectangle(
    (roi_min_vox[0], roi_min_vox[1]),
    roi_max_vox[0] - roi_min_vox[0],
    roi_max_vox[1] - roi_min_vox[1],
    linewidth=2, edgecolor='cyan', facecolor='none', linestyle='--'
)
ax.add_patch(rect)

ax.set_title(f'Axial Slice {mid_slice} of {sample_id}\nwith ROI Bounding Box', fontsize=14, fontweight='bold')
ax.set_xlabel('Right-Left (voxels)')
ax.set_ylabel('Posterior-Anterior (voxels)')
plt.tight_layout()
plt.show()

Now let's build a summary DataFrame by loading all annotation files to analyze the distribution of ROI sizes and image properties across the full dataset.

In [ ]:
# Load all annotation files and build a summary
records = []
for markup_obj in markup_files:
    key = markup_obj['Key']
    sid = key.split('/')[-1].replace('.mrk.json', '')
    resp = s3.get_object(Bucket=bucket, Key=key)
    ann = json.loads(resp['Body'].read())
    roi_info = ann['markups'][0]

    records.append({
        'sample_id': sid,
        'roi_center_L': roi_info['center'][0],
        'roi_center_P': roi_info['center'][1],
        'roi_center_S': roi_info['center'][2],
        'roi_size_L': roi_info['size'][0],
        'roi_size_P': roi_info['size'][1],
        'roi_size_S': roi_info['size'][2],
    })

df = polars.DataFrame(records)
df.head()

### Further Acknowledgments and Suggestions

This dataset provides paired 3D MRI volumes with corresponding 3D bounding box annotations, making it well-suited for benchmarking purposes for:

- **3D object detection** models that predict ROI bounding boxes from MRI input
- **anomaly localization** that tests localization models generalization/robustness under edge cases

Consider using [MONAI](https://monai.io/) (Medical Open Network for Artificial Intelligence), a PyTorch-based framework specifically designed for healthcare imaging deep learning, which has built-in support for NRRD loading and 3D detection tasks

We would like to thank NashBio organization for the data used in this benchmark. Curt Allen, Sarolta Bognár, Michail Fanariotis, Amin Honarmandi Shandiz, Andres Marcos Carrion, GEHC-AI annotation and real-world data teams for scoping and preparation of the dataset. 

 

### Cite this work as
Yang, Z., DSouza, N., Megyeri, I. et al. Decipher-MR: a vision-language foundation model for 3D MRI representations. npj Digit. Med. (2026). https://doi.org/10.1038/s41746-026-02596-4